# 04 — Model Evaluation & Comparison

Loads each trained checkpoint, evaluates on the held-out **test** split, and builds
the comparison table + confusion matrix required for the Milestone 1 deliverables.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
import pandas as pd
from torch.utils.data import DataLoader

from dataset import CelebASubset
from models import CustomCNN, build_resnet18, num_total_params
from utils import get_device, evaluate, plot_confusion_matrix, model_size_mb

REPO_ROOT = Path.cwd().parent
LOGS_DIR = REPO_ROOT / "logs"
device = get_device()

test_ds = CelebASubset("test")
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
num_classes = test_ds.num_classes

label_map = json.loads((REPO_ROOT / "data" / "splits" / "label_map.json").read_text())
class_names = [str(k) for k, v in sorted(label_map.items(), key=lambda kv: kv[1])]
print("classes:", class_names)


classes: ['3', '7', '1212', '8335']


## Register the runs to compare

Add/remove entries here to match whichever checkpoints you actually trained (custom CNN, ResNet18 fine-tuned, and optionally the frozen-backbone variant).

In [2]:
RUNS = {
    "custom_cnn": lambda: CustomCNN(num_classes=num_classes),
    "resnet18_finetune": lambda: build_resnet18(num_classes=num_classes, mode="finetune"),
    # "resnet18_feature_extract": lambda: build_resnet18(num_classes=num_classes, mode="feature_extract"),
}

results = {}
for run_name, build_fn in RUNS.items():
    ckpt_path = LOGS_DIR / f"{run_name}_best.pt"
    hist_path = LOGS_DIR / f"{run_name}_history.json"
    if not ckpt_path.exists():
        print(f"[skip] {run_name}: no checkpoint at {ckpt_path} (run its training notebook first)")
        continue

    model = build_fn()
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    eval_out = evaluate(model, test_loader, device, class_names=class_names)
    plot_confusion_matrix(eval_out["confusion_matrix"], class_names, run_name)

    hist = json.loads(hist_path.read_text()) if hist_path.exists() else {"train_time_sec": None}
    results[run_name] = {
        "test_accuracy": eval_out["accuracy"],
        "params": num_total_params(model),
        "size_mb": model_size_mb(model),
        "train_time_sec": hist.get("train_time_sec"),
        "best_val_acc": hist.get("best_val_acc"),
        "per_class_report": eval_out["report"],
    }
    print(f"{run_name}: test_acc={eval_out['accuracy']:.4f}")


custom_cnn: test_acc=0.5000
resnet18_finetune: test_acc=0.8750


## Comparison table (accuracy, size, training time)

In [3]:
rows = []
for run_name, r in results.items():
    rows.append({
        "model": run_name,
        "test_accuracy": round(r["test_accuracy"], 4),
        "val_accuracy": round(r["best_val_acc"], 4) if r["best_val_acc"] is not None else None,
        "num_params": r["params"],
        "size_mb": round(r["size_mb"], 2),
        "train_time_sec": round(r["train_time_sec"], 1) if r["train_time_sec"] else None,
    })

comparison_df = pd.DataFrame(rows).sort_values("test_accuracy", ascending=False)
comparison_df.to_csv(LOGS_DIR / "comparison_table.csv", index=False)
comparison_df


,model,test_accuracy,val_accuracy,num_params,size_mb,train_time_sec
1,resnet18_finetune,0.875,0.9375,11178564,42.64,461.3
0,custom_cnn,0.500,0.6875,1207588,4.61,927.1


## Per-class accuracy for the best model

In [4]:
best_run = comparison_df.iloc[0]["model"]
print("Best model by test accuracy:", best_run)

per_class = results[best_run]["per_class_report"]
per_class_df = pd.DataFrame(per_class).T
per_class_df.to_csv(LOGS_DIR / f"{best_run}_per_class_report.csv")
per_class_df


Best model by test accuracy: resnet18_finetune


,precision,recall,f1-score,support
3,0.800,1.000,0.888889,4.000
7,0.800,1.000,0.888889,4.000
1212,1.000,0.750,0.857143,4.000
8335,1.000,0.750,0.857143,4.000
accuracy,0.875,0.875,0.875000,0.875
macro avg,0.900,0.875,0.873016,16.000
weighted avg,0.900,0.875,0.873016,16.000


## Best-model justification

_Fill this in once real numbers are in `comparison_table.csv` — talking points to
cover (per the rubric): accuracy vs. the other model(s), parameter count / model
size, training time, and how much labeled data each approach needed to reach its
accuracy. Copy the final paragraph into `docs/training_results.md`._

Typical trade-off pattern to expect (confirm against your actual run):
- The **custom CNN** trains fastest per epoch and has far fewer parameters, but
  starting from random weights on a small (~150-200 image) dataset usually caps its
  accuracy lower than a pretrained backbone.
- **ResNet18 fine-tuned** brings ImageNet-pretrained features, typically reaching
  higher test accuracy with the same small dataset, at the cost of a much larger
  parameter count / model size and slower training.
- If you added the **frozen-backbone** ResNet18 variant, it usually lands between
  the two on training time (only the head trains) while its accuracy depends on how
  well ImageNet features transfer to face-identity discrimination without fine-tuning.

Carry the winning model's checkpoint (`logs/<run_name>_best.pt`) forward into
Milestone 2/3.